# Defaults screening for final alpha=0.5 hybrid/corr feature sets — minimal state (11 features)

This notebook reads the finalized train-only feature-selection artifacts from `hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2`, rebuilds z-window datasets through the standard feature pipeline, runs default-parameter bandit screening, and selects **top-1 corr** plus **top-1 hybrid** candidate per bandit algorithm for Optuna.

Minimal-state update: the bandit context uses `10 market features + 1 state feature = 11 total features`, where the only state feature is `state_in_position`.


## 0. Imports and project setup

In [1]:
import json
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"D:\PythonProjects\VKR")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_processing.functions.klines_dataloader import KlinesDataLoader
from data_processing.functions.stream_indicators import stream_TA_lib
from data_processing.functions.transform_indicators import transform_indicators_df
from data_processing.functions.rolling_z_score_clip import rolling_z_score_clip_df
from backtest.backtest_class import Backtesting

## 1. Configuration

In [2]:
# =====================================================================
# Core protocol
# =====================================================================
SEED = 142
ALL_SYMBOLS = ["BTCUSDT", "DOGEUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT"]
ACTIONS = [0, 1]

INTERVAL = 240
HORIZON = 10
TRADE_COST = 0.0025
OHLCV_PATH = PROJECT_ROOT / f"data/klines_data/crypto_{INTERVAL}m_bybit_TEST.parquet"

VAL_BARS = 2000
TEST_BARS = 2000
EMBARGO_BARS = 0

# Minimal state context: keep only position flag.
# Market feature sets still contain 10 selected market features, so bandit dimension is 10 + 1 = 11.
STATE_FEATURES = [
    "state_in_position",
]

META_COLS = ["timestamp", "symbol", "open", "high", "low", "close", "volume"]

# =====================================================================
# Source feature sets from final hybrid/corr alpha=0.5 feature-selection pipeline
# =====================================================================
FEATURE_SELECTION_ROOT = PROJECT_ROOT / "feature_selection" / "hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2"
FEATURE_SETS_JSON = FEATURE_SELECTION_ROOT / "feature_sets_hybrid_corr_stable_hie_alpha0p5.json"
FEATURE_META_JSON = FEATURE_SELECTION_ROOT / "feature_set_meta_hybrid_corr_stable_hie_alpha0p5.json"

OUTPUT_DIR = FEATURE_SELECTION_ROOT / "algorithm_specific_screening_defaults_state1_hmem250_ts30"
SCREENING_OUTPUT_DIR = OUTPUT_DIR
DIAGNOSTICS_OUTPUT_DIR = OUTPUT_DIR / "diagnostics"
for d in [OUTPUT_DIR, SCREENING_OUTPUT_DIR, DIAGNOSTICS_OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# =====================================================================
# Execution defaults fixed during screening
# =====================================================================
START_CAPITAL = 100.0
POSITION_SIZE = 0.10
MIN_HOLD_BARS = 4
COOLDOWN_BARS = 2
CONFIDENCE_THRESHOLD = 0.005
SCREENING_UPDATE_ON_VALIDATION = True

# =====================================================================
# Bandit defaults fixed during screening
# =====================================================================
# Previous short-memory HPO range for 13 total features was H_mem/W in [200, 400].
# With 11 total features we scale memory approximately by 11/13 to preserve
# observations-per-feature: [200, 400] * 11/13 = [169, 338].
# Rounded practical range: [170, 350] bars. For default screening use the
# rounded midpoint 250 bars. For discounted bandits H_mem ≈ 1 / (1 - gamma).
OLD_TOTAL_FEATURES_FOR_MEMORY = 13
NEW_TOTAL_FEATURES_FOR_MEMORY = 11
RECOMMENDED_MEMORY_RANGE_BARS = (170, 350)
DEFAULT_MEMORY_HORIZON_BARS = 325
DEFAULT_DISCOUNT_FACTOR = 1.0 - 1.0 / DEFAULT_MEMORY_HORIZON_BARS  # 0.996 for H_mem≈250
DEFAULT_WINDOW_SIZE = DEFAULT_MEMORY_HORIZON_BARS
DEFAULT_LAMBDA_PRIOR = 1.0
DEFAULT_NOISE_STD = 0.03
DEFAULT_UCB_ALPHA = 0.10
DEFAULT_REWARD_CLIP = 0.10

BANDIT_SCREENING_CONFIGS = {
    "discounted_lints": {
        "bandit_type": "discounted_lints",
        "discount_factor": DEFAULT_DISCOUNT_FACTOR,
        "lambda_prior": DEFAULT_LAMBDA_PRIOR,
        "noise_std": DEFAULT_NOISE_STD,
        "reward_clip": DEFAULT_REWARD_CLIP,
        "seed": SEED,
    },
    "discounted_linucb": {
        "bandit_type": "discounted_linucb",
        "discount_factor": DEFAULT_DISCOUNT_FACTOR,
        "lambda_prior": DEFAULT_LAMBDA_PRIOR,
        "ucb_alpha": DEFAULT_UCB_ALPHA,
        "reward_clip": DEFAULT_REWARD_CLIP,
        "seed": SEED,
    },
    "sw_lints": {
        "bandit_type": "sw_lints",
        "window_size": DEFAULT_WINDOW_SIZE,
        "lambda_prior": DEFAULT_LAMBDA_PRIOR,
        "noise_std": DEFAULT_NOISE_STD,
        "reward_clip": DEFAULT_REWARD_CLIP,
        "seed": SEED,
    },
    "sw_linucb": {
        "bandit_type": "sw_linucb",
        "window_size": DEFAULT_WINDOW_SIZE,
        "lambda_prior": DEFAULT_LAMBDA_PRIOR,
        "ucb_alpha": DEFAULT_UCB_ALPHA,
        "reward_clip": DEFAULT_REWARD_CLIP,
        "seed": SEED,
    },
}

TS_BANDITS = ["discounted_lints", "sw_lints"]
UCB_BANDITS = ["discounted_linucb", "sw_linucb"]

# Screening seeds. Change here only if compute budget requires it.
TS_SCREENING_SEEDS = list(range(142, 172))  # 30 seeds: 142..171
UCB_SCREENING_SEEDS = [SEED]

CONFIG_FOR_INDICATORS = {
    "ema_periods": [9, 21, 50, 100, 200],
    "momentum_indicators_periods": [14, 30, 72],
    "return_indicators_periods": [6, 24, 72, 168],
    "volatility_indicators_periods": [24, 72, 168],
    "level_periods": [24, 72, 168],
    "vol_ma_period": [24, 72, 168],
    "range_ma_period": [24, 72, 168],
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FEATURE_SELECTION_ROOT:", FEATURE_SELECTION_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("TS_SCREENING_SEEDS:", TS_SCREENING_SEEDS)
print("UCB_SCREENING_SEEDS:", UCB_SCREENING_SEEDS)
print("STATE_FEATURES:", STATE_FEATURES)
print("n_state_features:", len(STATE_FEATURES))
print("RECOMMENDED_MEMORY_RANGE_BARS:", RECOMMENDED_MEMORY_RANGE_BARS)
print("DEFAULT_MEMORY_HORIZON_BARS:", DEFAULT_MEMORY_HORIZON_BARS)
print("DEFAULT_DISCOUNT_FACTOR:", DEFAULT_DISCOUNT_FACTOR)
print("DEFAULT_WINDOW_SIZE:", DEFAULT_WINDOW_SIZE)
print("Bandits:", list(BANDIT_SCREENING_CONFIGS))

PROJECT_ROOT: D:\PythonProjects\VKR
FEATURE_SELECTION_ROOT: D:\PythonProjects\VKR\feature_selection\hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2
OUTPUT_DIR: D:\PythonProjects\VKR\feature_selection\hybrid_stable_hie_corr_outputs_alpha0p5_train_only_v2\algorithm_specific_screening_defaults_state1_hmem250_ts30
TS_SCREENING_SEEDS: [142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171]
UCB_SCREENING_SEEDS: [142]
STATE_FEATURES: ['state_in_position']
n_state_features: 1
RECOMMENDED_MEMORY_RANGE_BARS: (170, 350)
DEFAULT_MEMORY_HORIZON_BARS: 325
DEFAULT_DISCOUNT_FACTOR: 0.9969230769230769
DEFAULT_WINDOW_SIZE: 325
Bandits: ['discounted_lints', 'discounted_linucb', 'sw_lints', 'sw_linucb']


### Minimal-state and memory scaling note

This screening run uses only `state_in_position` from the state context. The market feature sets still have 10 features, so the bandit dimension is `10 + 1 = 11`.

The previous memory range `[200, 400]` was calibrated for 13 total features. To preserve approximately the same observations-per-feature ratio, the memory range is scaled by `11/13`: `[200, 400] × 11/13 ≈ [169, 338]`, rounded to `[170, 350]`. The default screening value is set to 250 bars, so `window_size = 250` and `discount_factor = 1 - 1/250 = 0.996`.


## 2. Load finalized hybrid/corr feature sets

In [3]:
if not FEATURE_SETS_JSON.exists():
    raise FileNotFoundError(f"FEATURE_SETS_JSON not found: {FEATURE_SETS_JSON}")
if not FEATURE_META_JSON.exists():
    raise FileNotFoundError(f"FEATURE_META_JSON not found: {FEATURE_META_JSON}")

with open(FEATURE_SETS_JSON, "r", encoding="utf-8") as f:
    FEATURE_SETS = json.load(f)
with open(FEATURE_META_JSON, "r", encoding="utf-8") as f:
    FEATURE_SET_META_RAW = json.load(f)

# Normalize metadata to the fields expected by the screening helpers.
FEATURE_SET_META = {}
for set_name, features in FEATURE_SETS.items():
    meta = dict(FEATURE_SET_META_RAW.get(set_name, {}))
    meta.setdefault("set_name", set_name)
    meta.setdefault("features", features)
    meta.setdefault("n_features", len(features))

    # Infer z_window and alpha if metadata is absent.
    if "z_window" not in meta:
        m = __import__("re").search(r"z(\d+)", set_name)
        if not m:
            raise ValueError(f"Cannot infer z_window from set name: {set_name}")
        meta["z_window"] = int(m.group(1))
    if "alpha_out" not in meta:
        if "a0p5" in set_name:
            meta["alpha_out"] = 0.5
        elif "a1" in set_name:
            meta["alpha_out"] = 1.0
        elif "a0" in set_name:
            meta["alpha_out"] = 0.0
        else:
            raise ValueError(f"Cannot infer alpha_out from set name: {set_name}")

    family = meta.get("family", "")
    if not family:
        family = "hybrid_corr_stable_hie" if "hybrid" in set_name else "corr_pruned"
    meta["family"] = family

    # strategy is kept for compatibility with previous screening aggregators.
    if "hybrid" in family or "hybrid" in set_name:
        meta["strategy"] = "hybrid_corr_stable_hie"
        meta["method_group"] = "hybrid"
    elif "corr" in family or "corr" in set_name:
        meta["strategy"] = "corr_pruned"
        meta["method_group"] = "corr"
    else:
        meta["strategy"] = family
        meta["method_group"] = family

    FEATURE_SET_META[set_name] = meta

screening_feature_sets = sorted(
    FEATURE_SETS.keys(),
    key=lambda name: (
        int(FEATURE_SET_META[name]["z_window"]),
        str(FEATURE_SET_META[name]["method_group"]),
        name,
    ),
)

screening_feature_sets_table = pd.DataFrame([
    {
        "set_name": name,
        "n_features_full": len(FEATURE_SETS[name]),
        "family": FEATURE_SET_META[name]["family"],
        "strategy": FEATURE_SET_META[name]["strategy"],
        "method_group": FEATURE_SET_META[name]["method_group"],
        "z_window": int(FEATURE_SET_META[name]["z_window"]),
        "alpha_out": float(FEATURE_SET_META[name]["alpha_out"]),
        "feature_list": "|".join(FEATURE_SETS[name]),
    }
    for name in screening_feature_sets
])

expected_n_sets = 8
if len(screening_feature_sets) != expected_n_sets:
    raise ValueError(f"Expected {expected_n_sets} feature sets, got {len(screening_feature_sets)}")
if not screening_feature_sets_table["n_features_full"].eq(10).all():
    display(screening_feature_sets_table[screening_feature_sets_table["n_features_full"] != 10])
    raise ValueError("Expected all feature sets to have 10 market features")
if not set(screening_feature_sets_table["method_group"]).issubset({"corr", "hybrid"}):
    display(screening_feature_sets_table)
    raise ValueError("Unexpected method_group values")

screening_feature_sets_table.to_csv(SCREENING_OUTPUT_DIR / "screening_feature_sets_table.csv", index=False)
print("Feature sets loaded:", len(screening_feature_sets))
display(screening_feature_sets_table)

Feature sets loaded: 8


,set_name,n_features_full,family,strategy,method_group,z_window,alpha_out,feature_list
0,z24_a0p5_corr_pruned_top10,10,corr_pruned,corr_pruned,corr,24,0.5,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...
1,z24_a0p5_hybrid_corr5_stablehie5_top10,10,hybrid_corr_stable_hie,hybrid_corr_stable_hie,hybrid,24,0.5,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...
2,z48_a0p5_corr_pruned_top10,10,corr_pruned,corr_pruned,corr,48,0.5,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...
3,z48_a0p5_hybrid_corr5_stablehie5_top10,10,hybrid_corr_stable_hie,hybrid_corr_stable_hie,hybrid,48,0.5,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...
4,z72_a0p5_corr_pruned_top10,10,corr_pruned,corr_pruned,corr,72,0.5,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...
5,z72_a0p5_hybrid_corr5_stablehie5_top10,10,hybrid_corr_stable_hie,hybrid_corr_stable_hie,hybrid,72,0.5,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...
6,z96_a0p5_corr_pruned_top10,10,corr_pruned,corr_pruned,corr,96,0.5,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...
7,z96_a0p5_hybrid_corr5_stablehie5_top10,10,hybrid_corr_stable_hie,hybrid_corr_stable_hie,hybrid,96,0.5,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...


## 3. Load OHLCV data through `KlinesDataLoader`

In [4]:
loader = KlinesDataLoader(symbols=ALL_SYMBOLS)

ohlcv_data = loader.load_data(
    download_path=str(OHLCV_PATH),
    analyse_data=True,
    cleaning=True,
)

ohlcv_data = (
    ohlcv_data
    .sort_values(["symbol", "timestamp"])
    .reset_index(drop=True)
)

print("OHLCV shape:", ohlcv_data.shape)
print("Date range:", ohlcv_data["timestamp"].min(), "->", ohlcv_data["timestamp"].max())
display(ohlcv_data["symbol"].value_counts())

			Статистика без фильтрации

Количество строк: 51672
Количество столбцов: 8
Количество пропущенных значений: 0

Название столбцов: timestamp, open, high, low, close, volume, turnover, symbol
Название активов: BTCUSDT, DOGEUSDT, ETHUSDT, SOLUSDT, XRPUSDT

Длина временного ряда актива BTCUSDT: 10549
Длина временного ряда актива DOGEUSDT: 10209
Длина временного ряда актива ETHUSDT: 10549
Длина временного ряда актива SOLUSDT: 9903
Длина временного ряда актива XRPUSDT: 10462

Временные рамки ряда каждого актива:
BTCUSDT: 2021-07-05 12:00:00+00:00 - 2026-04-28 12:00:00+00:00
DOGEUSDT: 2021-08-31 04:00:00+00:00 - 2026-04-28 12:00:00+00:00
ETHUSDT: 2021-07-05 12:00:00+00:00 - 2026-04-28 12:00:00+00:00
SOLUSDT: 2021-10-21 04:00:00+00:00 - 2026-04-28 12:00:00+00:00
XRPUSDT: 2021-07-20 00:00:00+00:00 - 2026-04-28 12:00:00+00:00


			Статистика после фильтрации (удаление столбца 'turnover' и выравнивание временных рядов по активам)

Количество строк: 49515
Количество столбцов: 7
Количество пропущ

symbol
BTCUSDT     9903
DOGEUSDT    9903
ETHUSDT     9903
SOLUSDT     9903
XRPUSDT     9903
Name: count, dtype: int64

## 4. Helper functions: feature engineering and chronological splits

In [5]:
def process_indicators_for_z_window(
    ohlcv_data: pd.DataFrame,
    symbols: list[str],
    meta_cols: list[str],
    indicator_config: dict,
    z_window: int,
    clip_value: float = 5.0,
    shift_by_one: bool = True,
) -> pd.DataFrame:
    processed_parts = []
    for symbol in symbols:
        print(f"Processing {symbol}, z_window={z_window}...")
        df_symbol = (
            ohlcv_data[ohlcv_data["symbol"] == symbol]
            .sort_values("timestamp")
            .reset_index(drop=True)
        )
        indicators_symbol = stream_TA_lib(
            df=df_symbol,
            meta_cols=meta_cols,
            **indicator_config,
        )
        indicators_symbol = indicators_symbol.dropna().sort_values("timestamp").reset_index(drop=True)
        transformed_symbol = transform_indicators_df(
            df=indicators_symbol,
            meta_cols=meta_cols,
        )
        z_score_symbol = rolling_z_score_clip_df(
            df=transformed_symbol,
            meta_cols=meta_cols,
            window=z_window,
            clip_value=clip_value,
            shift_by_one=shift_by_one,
        )
        z_score_symbol = z_score_symbol.dropna().sort_values("timestamp").reset_index(drop=True)
        processed_parts.append(z_score_symbol)

    out = (
        pd.concat(processed_parts, ignore_index=True)
        .sort_values(["symbol", "timestamp"])
        .reset_index(drop=True)
    )
    return out


def split_train_val_test_by_tail_bars(
    df: pd.DataFrame,
    symbols: list[str],
    val_bars: int,
    test_bars: int,
    embargo_bars: int = 0,
):
    train_parts, val_parts, test_parts, rows = [], [], [], []
    for sym in symbols:
        g = df[df["symbol"] == sym].sort_values("timestamp").reset_index(drop=True).copy()
        n = len(g)
        if n <= val_bars + test_bars + max(embargo_bars, 0):
            raise ValueError(f"Too few rows for {sym}: {n}")

        test_start = n - test_bars
        val_start = test_start - val_bars
        train_end = max(val_start - embargo_bars, 0)
        val_end = test_start - embargo_bars if embargo_bars > 0 else test_start

        train = g.iloc[:train_end].copy()
        val = g.iloc[val_start:val_end].copy()
        test = g.iloc[test_start:].copy()

        train_parts.append(train)
        val_parts.append(val)
        test_parts.append(test)
        rows.append({
            "symbol": sym,
            "n_total": n,
            "n_train": len(train),
            "n_val": len(val),
            "n_test": len(test),
            "train_min": train["timestamp"].min(),
            "train_max": train["timestamp"].max(),
            "val_min": val["timestamp"].min(),
            "val_max": val["timestamp"].max(),
            "test_min": test["timestamp"].min(),
            "test_max": test["timestamp"].max(),
        })

    split_summary = pd.DataFrame(rows)
    train_df = pd.concat(train_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    val_df = pd.concat(val_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    test_df = pd.concat(test_parts, ignore_index=True).sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    return train_df, val_df, test_df, split_summary


def get_feature_cols_for_selection(df: pd.DataFrame, meta_cols: list[str]):
    target_like_cols = [col for col in df.columns if col.startswith("target_log_ret_horizon_")]
    forbidden_cols = []
    for col in df.columns:
        if col.startswith("ema_") and not col.startswith(("ema_slope", "ema_spread")):
            forbidden_cols.append(col)
        if col.startswith(("vol_ma_", "range_ma_")):
            forbidden_cols.append(col)
        if col.startswith("MOM_") and not col.startswith("MOM_pct"):
            forbidden_cols.append(col)
        if col == "MACD_hist":
            forbidden_cols.append(col)

    feature_cols = [
        col for col in df.columns
        if col not in meta_cols
        and col not in target_like_cols
        and col not in forbidden_cols
    ]
    return feature_cols


def validate_features(df: pd.DataFrame, feature_cols: list[str]):
    if len(feature_cols) == 0:
        raise ValueError("feature_cols is empty")
    values = df[feature_cols].to_numpy(dtype=float)
    if np.isnan(values).any():
        raise ValueError("NaN in feature_cols")
    if np.isinf(values).any():
        raise ValueError("inf in feature_cols")
    zero_std = df[feature_cols].std() < 1e-12
    if zero_std.any():
        bad = zero_std[zero_std].index.tolist()
        raise ValueError(f"Zero-variance features: {bad}")
    print("Feature validation passed:", len(feature_cols))

## 5. Build z-window train/val/test datasets and validate selected features

In [6]:
DATASETS_BY_Z = {}
split_summaries = []

z_windows_to_build = sorted({int(FEATURE_SET_META[name]["z_window"]) for name in screening_feature_sets})
print("z_windows_to_build:", z_windows_to_build)

for z_window in z_windows_to_build:
    print("=" * 100)
    print(f"Building z_window={z_window}")
    print("=" * 100)

    processed = process_indicators_for_z_window(
        ohlcv_data=ohlcv_data,
        symbols=ALL_SYMBOLS,
        meta_cols=META_COLS,
        indicator_config=CONFIG_FOR_INDICATORS,
        z_window=z_window,
        clip_value=5.0,
        shift_by_one=True,
    )

    train_z, val_z, test_z, split_summary = split_train_val_test_by_tail_bars(
        df=processed,
        symbols=ALL_SYMBOLS,
        val_bars=VAL_BARS,
        test_bars=TEST_BARS,
        embargo_bars=EMBARGO_BARS,
    )
    split_summary.insert(0, "z_window", z_window)
    split_summaries.append(split_summary)

    feature_cols = get_feature_cols_for_selection(train_z, META_COLS)
    validate_features(train_z, feature_cols)

    # Validate that all selected features for this z-window exist in both train and validation frames.
    z_sets = [name for name in screening_feature_sets if int(FEATURE_SET_META[name]["z_window"]) == z_window]
    for set_name in z_sets:
        selected = FEATURE_SETS[set_name]
        missing_train = [f for f in selected if f not in train_z.columns]
        missing_val = [f for f in selected if f not in val_z.columns]
        if missing_train or missing_val:
            raise ValueError(
                f"Missing features for {set_name}: train={missing_train}, val={missing_val}"
            )

    DATASETS_BY_Z[z_window] = {
        "train": train_z,
        "val": val_z,
        "test": test_z,
        "feature_cols": feature_cols,
    }

split_summary_used = pd.concat(split_summaries, ignore_index=True)
split_summary_used.to_csv(DIAGNOSTICS_OUTPUT_DIR / "split_summary_used.csv", index=False)
display(split_summary_used)

z_windows_to_build: [24, 48, 72, 96]
Building z_window=24
Processing BTCUSDT, z_window=24...
Processing DOGEUSDT, z_window=24...
Processing ETHUSDT, z_window=24...
Processing SOLUSDT, z_window=24...
Processing XRPUSDT, z_window=24...
Feature validation passed: 54
Building z_window=48
Processing BTCUSDT, z_window=48...
Processing DOGEUSDT, z_window=48...
Processing ETHUSDT, z_window=48...
Processing SOLUSDT, z_window=48...
Processing XRPUSDT, z_window=48...
Feature validation passed: 54
Building z_window=72
Processing BTCUSDT, z_window=72...
Processing DOGEUSDT, z_window=72...
Processing ETHUSDT, z_window=72...
Processing SOLUSDT, z_window=72...
Processing XRPUSDT, z_window=72...
Feature validation passed: 54
Building z_window=96
Processing BTCUSDT, z_window=96...
Processing DOGEUSDT, z_window=96...
Processing ETHUSDT, z_window=96...
Processing SOLUSDT, z_window=96...
Processing XRPUSDT, z_window=96...
Feature validation passed: 54


,z_window,symbol,n_total,n_train,n_val,n_test,train_min,train_max,val_min,val_max,test_min,test_max
0,24,BTCUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
1,24,DOGEUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
2,24,ETHUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
3,24,SOLUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
4,24,XRPUSDT,9674,5674,2000,2000,2021-11-28 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
5,48,BTCUSDT,9650,5650,2000,2000,2021-12-02 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
6,48,DOGEUSDT,9650,5650,2000,2000,2021-12-02 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
7,48,ETHUSDT,9650,5650,2000,2000,2021-12-02 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
8,48,SOLUSDT,9650,5650,2000,2000,2021-12-02 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00
9,48,XRPUSDT,9650,5650,2000,2000,2021-12-02 08:00:00+00:00,2024-06-30 20:00:00+00:00,2024-07-01 00:00:00+00:00,2025-05-30 04:00:00+00:00,2025-05-30 08:00:00+00:00,2026-04-28 12:00:00+00:00


## 6. Backtest metric and diagnostics helpers

In [7]:
def _get_store(backtesting, mode: str, store_name: str):
    if mode == "train":
        return getattr(backtesting, f"{store_name}_train")
    if mode == "val":
        return getattr(backtesting, f"{store_name}_val")
    raise ValueError("mode должен быть 'train' или 'val'")


def extract_trade_stats_from_log(trade_log: list[dict], trade_cost: float) -> pd.DataFrame:
    if not trade_log:
        return pd.DataFrame()
    events = pd.DataFrame(trade_log).copy()
    if events.empty or "event" not in events.columns:
        return pd.DataFrame()
    trades = []
    open_trade = None
    for _, row in events.sort_values("timestamp").iterrows():
        if row["event"] == "BUY":
            open_trade = row
        elif row["event"] == "SELL" and open_trade is not None:
            entry_price = float(open_trade["price"])
            exit_price = float(row["price"])
            pnl_before_cost = np.log(exit_price / entry_price)
            pnl_after_cost = pnl_before_cost + 2.0 * np.log(1.0 - trade_cost)
            trades.append({
                "entry_timestamp": open_trade["timestamp"],
                "exit_timestamp": row["timestamp"],
                "entry_price": entry_price,
                "exit_price": exit_price,
                "pnl_before_cost": pnl_before_cost,
                "pnl_after_cost": pnl_after_cost,
                "holding_time": pd.to_datetime(row["timestamp"]) - pd.to_datetime(open_trade["timestamp"]),
            })
            open_trade = None
    return pd.DataFrame(trades)


def compute_backtest_metrics(backtesting, sym: str, mode: str, trade_cost: float) -> dict:
    balance_store = _get_store(backtesting, mode, "balance")
    actions_store = _get_store(backtesting, mode, "actions")
    raw_actions_store = _get_store(backtesting, mode, "raw_actions")
    decision_log_store = _get_store(backtesting, mode, "decision_log")
    trade_log_store = _get_store(backtesting, mode, "trade_log")
    reward_log_store = _get_store(backtesting, mode, "reward_log")

    balance = np.asarray(balance_store.get(sym, []), dtype=float)
    actions = np.asarray(actions_store.get(sym, []), dtype=float)
    raw_actions = np.asarray(raw_actions_store.get(sym, []), dtype=float)
    decision_log = pd.DataFrame(decision_log_store.get(sym, []))
    trade_log = trade_log_store.get(sym, [])
    reward_log = pd.DataFrame(reward_log_store.get(sym, []))

    if len(balance) == 0:
        return {"symbol": sym, "mode": mode, "n_decisions": 0}

    start_value = float(balance[0])
    end_value = float(balance[-1])
    return_pct = (end_value / start_value - 1.0) * 100.0
    running_max = np.maximum.accumulate(balance)
    drawdown = balance / (running_max + 1e-12) - 1.0
    max_drawdown_pct = float(drawdown.min() * 100.0)

    rows = {
        "symbol": sym,
        "mode": mode,
        "start_value": start_value,
        "end_value": end_value,
        "return_pct": return_pct,
        "max_drawdown_pct": max_drawdown_pct,
        "drawdown_abs": abs(max_drawdown_pct),
        "raw_action_1_ratio": float(np.mean(raw_actions == 1)) if len(raw_actions) else np.nan,
        "executed_action_1_ratio": float(np.mean(actions == 1)) if len(actions) else np.nan,
        "executed_action_1": int(np.sum(actions == 1)) if len(actions) else 0,
        "raw_action_1": int(np.sum(raw_actions == 1)) if len(raw_actions) else 0,
        "n_decisions": int(len(actions)),
    }

    if not decision_log.empty:
        rows.update({
            "threshold_applied_ratio": float(decision_log["threshold_applied"].mean()),
            "constraint_applied_ratio": float(decision_log["constraint_applied"].mean()),
            "abs_edge_mean": float(decision_log["abs_edge"].mean()),
            "abs_edge_p25": float(decision_log["abs_edge"].quantile(0.25)),
            "abs_edge_p50": float(decision_log["abs_edge"].quantile(0.50)),
            "abs_edge_p75": float(decision_log["abs_edge"].quantile(0.75)),
            "abs_edge_p90": float(decision_log["abs_edge"].quantile(0.90)),
            "uncertainty_0_mean": float(decision_log["uncertainty_0"].mean()),
            "uncertainty_1_mean": float(decision_log["uncertainty_1"].mean()),
            "score_0_std": float(decision_log["score_0"].std()),
            "score_1_std": float(decision_log["score_1"].std()),
        })
    else:
        rows.update({
            "threshold_applied_ratio": np.nan,
            "constraint_applied_ratio": np.nan,
            "abs_edge_mean": np.nan,
            "abs_edge_p25": np.nan,
            "abs_edge_p50": np.nan,
            "abs_edge_p75": np.nan,
            "abs_edge_p90": np.nan,
            "uncertainty_0_mean": np.nan,
            "uncertainty_1_mean": np.nan,
            "score_0_std": np.nan,
            "score_1_std": np.nan,
        })

    if not reward_log.empty and "reward" in reward_log.columns:
        rew = reward_log["reward"].astype(float)
        rows.update({
            "n_reward_updates": int(len(reward_log)),
            "mean_reward_update": float(rew.mean()),
            "std_reward_update": float(rew.std()),
            "reward_abs_p50": float(rew.abs().quantile(0.50)),
            "reward_abs_p90": float(rew.abs().quantile(0.90)),
            "reward_abs_p95": float(rew.abs().quantile(0.95)),
            "reward_p05": float(rew.quantile(0.05)),
            "reward_p95": float(rew.quantile(0.95)),
        })
    else:
        rows.update({
            "n_reward_updates": 0,
            "mean_reward_update": np.nan,
            "std_reward_update": np.nan,
            "reward_abs_p50": np.nan,
            "reward_abs_p90": np.nan,
            "reward_abs_p95": np.nan,
            "reward_p05": np.nan,
            "reward_p95": np.nan,
        })

    trades = extract_trade_stats_from_log(trade_log=trade_log, trade_cost=trade_cost)
    if not trades.empty:
        gross_profit = trades.loc[trades["pnl_after_cost"] > 0, "pnl_after_cost"].sum()
        gross_loss = trades.loc[trades["pnl_after_cost"] < 0, "pnl_after_cost"].sum()
        rows.update({
            "trades": int(len(trades)),
            "win_rate": float((trades["pnl_after_cost"] > 0).mean()),
            "mean_trade_pnl": float(trades["pnl_after_cost"].mean()),
            "median_trade_pnl": float(trades["pnl_after_cost"].median()),
            "total_trade_pnl": float(trades["pnl_after_cost"].sum()),
            "profit_factor": float(gross_profit / abs(gross_loss + 1e-12)),
            "avg_holding": trades["holding_time"].mean(),
        })
    else:
        rows.update({
            "trades": 0,
            "win_rate": np.nan,
            "mean_trade_pnl": np.nan,
            "median_trade_pnl": np.nan,
            "total_trade_pnl": 0.0,
            "profit_factor": np.nan,
            "avg_holding": pd.NaT,
        })
    return rows


def summarize_validation_results(raw_results: pd.DataFrame) -> pd.DataFrame:
    val = raw_results[raw_results["mode"] == "val"].copy()
    if val.empty:
        return pd.DataFrame()
    val["profit_factor_clean"] = val["profit_factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    val["is_active"] = val["trades"] > 0
    val["has_long_actions"] = val["executed_action_1"] > 0
    val["is_positive_return"] = val["return_pct"] > 0
    summary = (
        val.groupby(["bandit_name", "set_name", "seed"], dropna=False)
        .agg(
            family=("family", "first"),
            strategy=("strategy", "first"),
            z_window=("z_window", "first"),
            alpha_out=("alpha_out", "first"),
            n_market_features=("n_market_features", "first"),
            n_symbols=("symbol", "nunique"),
            mean_return_pct=("return_pct", "mean"),
            median_return_pct=("return_pct", "median"),
            min_return_pct=("return_pct", "min"),
            max_return_pct=("return_pct", "max"),
            max_drawdown_abs=("drawdown_abs", "max"),
            mean_drawdown_abs=("drawdown_abs", "mean"),
            median_profit_factor=("profit_factor_clean", "median"),
            mean_profit_factor=("profit_factor_clean", "mean"),
            total_trades=("trades", "sum"),
            mean_trades=("trades", "mean"),
            mean_win_rate=("win_rate", "mean"),
            mean_raw_action_1_ratio=("raw_action_1_ratio", "mean"),
            mean_executed_action_1_ratio=("executed_action_1_ratio", "mean"),
            mean_threshold_applied_ratio=("threshold_applied_ratio", "mean"),
            mean_constraint_applied_ratio=("constraint_applied_ratio", "mean"),
            active_symbols=("is_active", "sum"),
            long_action_symbols=("has_long_actions", "sum"),
            positive_return_symbols=("is_positive_return", "sum"),
            feature_list=("feature_list", "first"),
        )
        .reset_index()
    )
    summary["all_symbols_active"] = summary["active_symbols"] == summary["n_symbols"]
    summary["all_symbols_have_long"] = summary["long_action_symbols"] == summary["n_symbols"]
    summary = summary.sort_values(
        [
            "bandit_name", "all_symbols_active", "all_symbols_have_long",
            "median_return_pct", "mean_return_pct", "max_drawdown_abs",
            "median_profit_factor", "total_trades",
        ],
        ascending=[True, False, False, False, False, True, False, False],
    ).reset_index(drop=True)
    summary["rank_within_bandit"] = summary.groupby("bandit_name").cumcount() + 1
    return summary


def collect_reward_and_decision_diagnostics(bt, run_meta: dict):
    reward_rows, decision_rows, bandit_diag_rows = [], [], []
    for phase in ["train", "val"]:
        reward_log = bt.get_reward_log_frame(phase) if hasattr(bt, "get_reward_log_frame") else pd.DataFrame()
        decision_log = bt.get_decision_log_frame(phase) if hasattr(bt, "get_decision_log_frame") else pd.DataFrame()

        if not reward_log.empty:
            group_cols = ["symbol", "raw_action", "decision_regime"]
            for keys, g in reward_log.groupby(group_cols, dropna=False):
                reward = g["reward"].astype(float)
                reward_rows.append({
                    **run_meta,
                    "phase": phase,
                    "symbol": keys[0],
                    "raw_action": keys[1],
                    "decision_regime": keys[2],
                    "n": len(g),
                    "reward_mean": float(reward.mean()),
                    "reward_std": float(reward.std()),
                    "reward_abs_p50": float(reward.abs().quantile(0.50)),
                    "reward_abs_p75": float(reward.abs().quantile(0.75)),
                    "reward_abs_p90": float(reward.abs().quantile(0.90)),
                    "reward_abs_p95": float(reward.abs().quantile(0.95)),
                    "reward_p05": float(reward.quantile(0.05)),
                    "reward_p95": float(reward.quantile(0.95)),
                    "positive_reward_ratio": float((reward > 0).mean()),
                    "future_log_ret_std": float(g["future_log_ret"].astype(float).std()),
                    "cost_applied_mean": float(g["cost_applied"].astype(float).mean()) if "cost_applied" in g else np.nan,
                })

        if not decision_log.empty:
            for sym, g in decision_log.groupby("symbol", dropna=False):
                decision_rows.append({
                    **run_meta,
                    "phase": phase,
                    "symbol": sym,
                    "n": len(g),
                    "abs_edge_mean": float(g["abs_edge"].mean()),
                    "abs_edge_p10": float(g["abs_edge"].quantile(0.10)),
                    "abs_edge_p25": float(g["abs_edge"].quantile(0.25)),
                    "abs_edge_p50": float(g["abs_edge"].quantile(0.50)),
                    "abs_edge_p75": float(g["abs_edge"].quantile(0.75)),
                    "abs_edge_p90": float(g["abs_edge"].quantile(0.90)),
                    "threshold_applied_ratio": float(g["threshold_applied"].mean()),
                    "constraint_applied_ratio": float(g["constraint_applied"].mean()),
                    "raw_action_1_ratio": float((g["raw_action"] == 1).mean()),
                    "executed_action_1_ratio": float((g["executed_action"] == 1).mean()),
                    "uncertainty_0_mean": float(g["uncertainty_0"].mean()),
                    "uncertainty_1_mean": float(g["uncertainty_1"].mean()),
                    "score_0_std": float(g["score_0"].std()),
                    "score_1_std": float(g["score_1"].std()),
                    "mean_0_std": float(g["mean_0"].std()),
                    "mean_1_std": float(g["mean_1"].std()),
                })

    if hasattr(bt, "get_bandit_diagnostics_frame"):
        diag = bt.get_bandit_diagnostics_frame()
        if not diag.empty:
            for row in diag.to_dict("records"):
                bandit_diag_rows.append({**run_meta, **row})

    return reward_rows, decision_rows, bandit_diag_rows

## 7. Screening runner

In [8]:
def run_screening_for_configs(
    bandit_configs: dict,
    feature_set_names: list[str],
    seeds_by_bandit: dict[str, list[int]],
    output_label: str,
):
    rows = []
    errors = []
    reward_diag_rows = []
    decision_diag_rows = []
    bandit_diag_rows = []

    total_runs = sum(len(seeds_by_bandit[b]) * len(feature_set_names) for b in bandit_configs)
    run_idx = 0

    for bandit_name, bandit_base_config in bandit_configs.items():
        for set_name in feature_set_names:
            meta = FEATURE_SET_META[set_name]
            z_window = int(meta["z_window"])
            alpha_out_for_backtest = float(meta["alpha_out"])
            selected_features = FEATURE_SETS[set_name]
            train_z = DATASETS_BY_Z[z_window]["train"]
            val_z = DATASETS_BY_Z[z_window]["val"]

            missing_train = [f for f in selected_features if f not in train_z.columns]
            missing_val = [f for f in selected_features if f not in val_z.columns]
            if missing_train or missing_val:
                errors.append({
                    "bandit_name": bandit_name,
                    "set_name": set_name,
                    "error": "missing_features",
                    "missing_train": "|".join(missing_train),
                    "missing_val": "|".join(missing_val),
                })
                continue

            for seed in seeds_by_bandit[bandit_name]:
                run_idx += 1
                print("=" * 120)
                print(f"[{run_idx}/{total_runs}] {output_label} | bandit={bandit_name} | seed={seed} | set={set_name}")
                print("=" * 120)

                config_for_bandit = dict(bandit_base_config)
                config_for_bandit["n_features"] = len(selected_features) + len(STATE_FEATURES)
                config_for_bandit["actions"] = ACTIONS
                config_for_bandit["seed"] = int(seed)

                run_meta = {
                    "run_label": output_label,
                    "bandit_name": bandit_name,
                    "set_name": set_name,
                    "seed": int(seed),
                    "family": meta["family"],
                    "strategy": meta["strategy"],
                    "z_window": z_window,
                    "alpha_out": alpha_out_for_backtest,
                    "n_market_features": len(selected_features),
                    "n_state_features": len(STATE_FEATURES),
                    "n_total_features": len(selected_features) + len(STATE_FEATURES),
                    "feature_list": "|".join(selected_features),
                    "update_on_validation": SCREENING_UPDATE_ON_VALIDATION,
                    "confidence_threshold": CONFIDENCE_THRESHOLD,
                    "min_hold_bars": MIN_HOLD_BARS,
                    "cooldown_bars": COOLDOWN_BARS,
                    "position_size": POSITION_SIZE,
                }

                try:
                    bt = Backtesting(
                        meta_cols=META_COLS,
                        feature_columns=selected_features,
                        config_for_bandit=config_for_bandit,
                        trade_cost=TRADE_COST,
                        seed=int(seed),
                        update_on_validation=SCREENING_UPDATE_ON_VALIDATION,
                        horizon=HORIZON,
                        min_hold_bars=MIN_HOLD_BARS,
                        cooldown_bars=COOLDOWN_BARS,
                        confidence_threshold=CONFIDENCE_THRESHOLD,
                        alpha_out=alpha_out_for_backtest,
                        state_feature_columns=STATE_FEATURES,
                    )
                    bt.backtest(
                        dataframe_train=train_z,
                        dataframe_val=val_z,
                        symbols=ALL_SYMBOLS,
                        start_capital=START_CAPITAL,
                        position_size=POSITION_SIZE,
                    )

                    for sym in ALL_SYMBOLS:
                        for mode in ["train", "val"]:
                            metrics = compute_backtest_metrics(bt, sym, mode, TRADE_COST)
                            metrics.update(run_meta)
                            for k, v in bandit_base_config.items():
                                metrics[f"bandit_param_{k}"] = v
                            rows.append(metrics)

                    rdiag, ddiag, bdiag = collect_reward_and_decision_diagnostics(bt, run_meta)
                    reward_diag_rows.extend(rdiag)
                    decision_diag_rows.extend(ddiag)
                    bandit_diag_rows.extend(bdiag)

                except Exception as e:
                    err = {**run_meta, "error": repr(e)}
                    errors.append(err)
                    print("ERROR:", err)

    return {
        "results": pd.DataFrame(rows),
        "errors": pd.DataFrame(errors),
        "reward_diagnostics": pd.DataFrame(reward_diag_rows),
        "decision_diagnostics": pd.DataFrame(decision_diag_rows),
        "bandit_diagnostics": pd.DataFrame(bandit_diag_rows),
    }

## 8. Run default-parameter screening

In [9]:
stage1_seeds_by_bandit = {
    bandit_name: (TS_SCREENING_SEEDS if bandit_name in TS_BANDITS else UCB_SCREENING_SEEDS)
    for bandit_name in BANDIT_SCREENING_CONFIGS
}

expected_screening_backtests = sum(
    len(stage1_seeds_by_bandit[b]) * len(screening_feature_sets)
    for b in BANDIT_SCREENING_CONFIGS
)
print("Expected screening backtests:", expected_screening_backtests)
print(json.dumps(stage1_seeds_by_bandit, ensure_ascii=False, indent=2))

stage1 = run_screening_for_configs(
    bandit_configs=BANDIT_SCREENING_CONFIGS,
    feature_set_names=screening_feature_sets,
    seeds_by_bandit=stage1_seeds_by_bandit,
    output_label="defaults_screening_alpha0p5_corr_vs_hybrid_ts20",
)

stage1_results = stage1["results"]
stage1_errors = stage1["errors"]
stage1_reward_diagnostics = stage1["reward_diagnostics"]
stage1_decision_diagnostics = stage1["decision_diagnostics"]
stage1_bandit_diagnostics = stage1["bandit_diagnostics"]

stage1_results.to_csv(SCREENING_OUTPUT_DIR / "stage1_screening_results_raw.csv", index=False)
stage1_errors.to_csv(SCREENING_OUTPUT_DIR / "stage1_screening_errors.csv", index=False)
stage1_reward_diagnostics.to_csv(DIAGNOSTICS_OUTPUT_DIR / "stage1_backtest_reward_diagnostics.csv", index=False)
stage1_decision_diagnostics.to_csv(DIAGNOSTICS_OUTPUT_DIR / "stage1_decision_edge_diagnostics.csv", index=False)
stage1_bandit_diagnostics.to_csv(DIAGNOSTICS_OUTPUT_DIR / "stage1_bandit_diagnostics.csv", index=False)

print("Stage1 results shape:", stage1_results.shape)
print("Stage1 errors:", len(stage1_errors))
display(stage1_results.head())
display(stage1_errors)

Expected screening backtests: 496
{
  "discounted_lints": [
    142,
    143,
    144,
    145,
    146,
    147,
    148,
    149,
    150,
    151,
    152,
    153,
    154,
    155,
    156,
    157,
    158,
    159,
    160,
    161,
    162,
    163,
    164,
    165,
    166,
    167,
    168,
    169,
    170,
    171
  ],
  "discounted_linucb": [
    142
  ],
  "sw_lints": [
    142,
    143,
    144,
    145,
    146,
    147,
    148,
    149,
    150,
    151,
    152,
    153,
    154,
    155,
    156,
    157,
    158,
    159,
    160,
    161,
    162,
    163,
    164,
    165,
    166,
    167,
    168,
    169,
    170,
    171
  ],
  "sw_linucb": [
    142
  ]
}
[1/496] defaults_screening_alpha0p5_corr_vs_hybrid_ts20 | bandit=discounted_lints | seed=142 | set=z24_a0p5_corr_pruned_top10
BTCUSDT: фаза train началась: 2021-11-28 08:00:00+00:00
BTCUSDT: фаза train закончилась: 2024-06-30 20:00:00+00:00
BTCUSDT: фаза val началась: 2024-07-01 00:00:00+00:00
BTCUSDT: фаз

,symbol,mode,start_value,end_value,return_pct,max_drawdown_pct,drawdown_abs,raw_action_1_ratio,executed_action_1_ratio,executed_action_1,...,cooldown_bars,position_size,bandit_param_bandit_type,bandit_param_discount_factor,bandit_param_lambda_prior,bandit_param_noise_std,bandit_param_reward_clip,bandit_param_seed,bandit_param_ucb_alpha,bandit_param_window_size
0,BTCUSDT,train,99.950063,84.948290,-15.009268,-17.672274,17.672274,0.471625,0.544413,3089,...,2,0.1,discounted_lints,0.996923,1.0,0.03,0.1,142,NaN,NaN
1,BTCUSDT,val,100.000000,97.447976,-2.552024,-4.214332,4.214332,0.477500,0.533500,1067,...,2,0.1,discounted_lints,0.996923,1.0,0.03,0.1,142,NaN,NaN
2,DOGEUSDT,train,99.950063,82.849533,-17.109074,-21.056366,21.056366,0.431265,0.512337,2907,...,2,0.1,discounted_lints,0.996923,1.0,0.03,0.1,142,NaN,NaN
3,DOGEUSDT,val,100.000000,101.762627,1.762627,-5.497870,5.497870,0.373000,0.449000,898,...,2,0.1,discounted_lints,0.996923,1.0,0.03,0.1,142,NaN,NaN
4,ETHUSDT,train,99.950063,79.227909,-20.732507,-22.849032,22.849032,0.459817,0.531900,3018,...,2,0.1,discounted_lints,0.996923,1.0,0.03,0.1,142,NaN,NaN


""


## 9. Ranking and balanced top-1 corr/hybrid selection per algorithm

In [10]:
stage1_seed_level_summary = summarize_validation_results(stage1_results)
stage1_seed_level_summary.to_csv(SCREENING_OUTPUT_DIR / "stage1_seed_level_validation_summary.csv", index=False)
display(stage1_seed_level_summary)


def aggregate_seed_level_ranking(seed_level_summary: pd.DataFrame, output_rank_col: str = "rank_within_bandit") -> pd.DataFrame:
    """
    Aggregate validation summaries across seeds.

    Ranking hierarchy is conservative:
    1. all symbols active share
    2. all symbols have long actions share
    3. median validation median_return_pct across seeds
    4. mean validation median_return_pct across seeds
    5. lower max drawdown
    6. profit factor
    7. total trades
    """
    if seed_level_summary.empty:
        return pd.DataFrame()

    out = (
        seed_level_summary
        .groupby(["bandit_name", "set_name"], dropna=False)
        .agg(
            family=("family", "first"),
            strategy=("strategy", "first"),
            z_window=("z_window", "first"),
            alpha_out=("alpha_out", "first"),
            n_market_features=("n_market_features", "first"),
            n_seeds=("seed", "nunique"),
            seeds_used=("seed", lambda x: "|".join(str(int(v)) for v in sorted(set(x)))),

            median_of_seed_median_return_pct=("median_return_pct", "median"),
            mean_of_seed_median_return_pct=("median_return_pct", "mean"),
            std_of_seed_median_return_pct=("median_return_pct", "std"),
            min_seed_median_return_pct=("median_return_pct", "min"),
            max_seed_median_return_pct=("median_return_pct", "max"),

            median_of_seed_mean_return_pct=("mean_return_pct", "median"),
            median_max_drawdown_abs=("max_drawdown_abs", "median"),
            median_profit_factor=("median_profit_factor", "median"),
            median_total_trades=("total_trades", "median"),
            median_positive_return_symbols=("positive_return_symbols", "median"),
            median_active_symbols=("active_symbols", "median"),
            median_long_action_symbols=("long_action_symbols", "median"),

            all_symbols_active_share=("all_symbols_active", "mean"),
            all_symbols_have_long_share=("all_symbols_have_long", "mean"),
            feature_list=("feature_list", "first"),
        )
        .reset_index()
    )

    out["std_of_seed_median_return_pct"] = out["std_of_seed_median_return_pct"].fillna(0.0)
    out["method_group"] = out["set_name"].map(lambda s: FEATURE_SET_META[s]["method_group"])

    out = (
        out
        .sort_values(
            [
                "bandit_name",
                "all_symbols_active_share",
                "all_symbols_have_long_share",
                "median_of_seed_median_return_pct",
                "mean_of_seed_median_return_pct",
                "median_max_drawdown_abs",
                "median_profit_factor",
                "median_total_trades",
            ],
            ascending=[True, False, False, False, False, True, False, False],
        )
        .reset_index(drop=True)
    )
    out[output_rank_col] = out.groupby("bandit_name").cumcount() + 1
    return out


stage1_ranking = aggregate_seed_level_ranking(stage1_seed_level_summary, output_rank_col="rank_within_bandit")
stage1_ranking.to_csv(SCREENING_OUTPUT_DIR / "stage1_set_validation_ranking.csv", index=False)
display(stage1_ranking)

# Balanced selection for Optuna: top-1 corr and top-1 hybrid per algorithm.
selected_feature_sets_by_bandit = {}
selected_rows = []

for bandit_name, part in stage1_ranking.groupby("bandit_name", sort=False):
    bandit_selected = []
    for method_group in ["corr", "hybrid"]:
        method_part = part[part["method_group"].eq(method_group)].sort_values("rank_within_bandit")
        if method_part.empty:
            raise ValueError(f"No {method_group} candidates for {bandit_name}")
        row = method_part.iloc[0].to_dict()
        set_name = row["set_name"]
        source = f"top1_{method_group}_within_{bandit_name}_defaults_screening"
        item = {
            "set_name": set_name,
            "method_group": method_group,
            "source": source,
            "rank_within_bandit": int(row["rank_within_bandit"]),
            "median_of_seed_median_return_pct": float(row["median_of_seed_median_return_pct"]),
            "mean_of_seed_median_return_pct": float(row["mean_of_seed_median_return_pct"]),
            "median_max_drawdown_abs": float(row["median_max_drawdown_abs"]),
            "median_profit_factor": float(row["median_profit_factor"]),
            "median_total_trades": float(row["median_total_trades"]),
            "n_seeds": int(row["n_seeds"]),
        }
        bandit_selected.append(item)
        selected_rows.append({"bandit_name": bandit_name, **item, **FEATURE_SET_META[set_name]})
    selected_feature_sets_by_bandit[bandit_name] = bandit_selected

selected_feature_sets_table = pd.DataFrame(selected_rows)
selected_feature_sets_table.to_csv(
    SCREENING_OUTPUT_DIR / "selected_top1_corr_and_hybrid_by_bandit_after_screening.csv",
    index=False,
)

with open(SCREENING_OUTPUT_DIR / "selected_top1_corr_and_hybrid_by_bandit_after_screening.json", "w", encoding="utf-8") as f:
    json.dump(selected_feature_sets_by_bandit, f, ensure_ascii=False, indent=2)

# Convenience export for Optuna: includes actual feature lists and normalized metadata.
selected_for_optuna = {}
for bandit_name, items in selected_feature_sets_by_bandit.items():
    selected_for_optuna[bandit_name] = []
    for item in items:
        set_name = item["set_name"]
        selected_for_optuna[bandit_name].append({
            **item,
            "features": FEATURE_SETS[set_name],
            "meta": FEATURE_SET_META[set_name],
        })

with open(SCREENING_OUTPUT_DIR / "selected_feature_sets_for_optuna_balanced.json", "w", encoding="utf-8") as f:
    json.dump(selected_for_optuna, f, ensure_ascii=False, indent=2)

print("Balanced selected feature sets by bandit:")
print(json.dumps(selected_feature_sets_by_bandit, ensure_ascii=False, indent=2))
display(selected_feature_sets_table)

,bandit_name,set_name,seed,family,strategy,z_window,alpha_out,n_market_features,n_symbols,mean_return_pct,...,mean_executed_action_1_ratio,mean_threshold_applied_ratio,mean_constraint_applied_ratio,active_symbols,long_action_symbols,positive_return_symbols,feature_list,all_symbols_active,all_symbols_have_long,rank_within_bandit
0,discounted_lints,z24_a0p5_corr_pruned_top10,148,corr_pruned,corr_pruned,24,0.5,10,5,1.043325,...,0.5236,0.0786,0.1407,5,5,3,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,True,True,1
1,discounted_lints,z24_a0p5_hybrid_corr5_stablehie5_top10,156,hybrid_corr_stable_hie,hybrid_corr_stable_hie,24,0.5,10,5,0.471477,...,0.4967,0.0803,0.1195,5,5,3,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,True,True,2
2,discounted_lints,z96_a0p5_hybrid_corr5_stablehie5_top10,143,hybrid_corr_stable_hie,hybrid_corr_stable_hie,96,0.5,10,5,0.583074,...,0.5000,0.0716,0.1231,5,5,3,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...,True,True,3
3,discounted_lints,z24_a0p5_hybrid_corr5_stablehie5_top10,146,hybrid_corr_stable_hie,hybrid_corr_stable_hie,24,0.5,10,5,3.453623,...,0.4904,0.0743,0.1151,5,5,3,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,True,True,4
4,discounted_lints,z24_a0p5_corr_pruned_top10,144,corr_pruned,corr_pruned,24,0.5,10,5,0.527339,...,0.5117,0.0803,0.1465,5,5,3,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,True,True,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491,sw_linucb,z24_a0p5_corr_pruned_top10,142,corr_pruned,corr_pruned,24,0.5,10,5,-1.887682,...,0.4985,0.0638,0.1227,5,5,1,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,True,True,4
492,sw_linucb,z48_a0p5_corr_pruned_top10,142,corr_pruned,corr_pruned,48,0.5,10,5,-4.725248,...,0.5015,0.0626,0.1339,5,5,1,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...,True,True,5
493,sw_linucb,z96_a0p5_corr_pruned_top10,142,corr_pruned,corr_pruned,96,0.5,10,5,-4.690513,...,0.4938,0.0499,0.1072,5,5,1,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...,True,True,6
494,sw_linucb,z96_a0p5_hybrid_corr5_stablehie5_top10,142,hybrid_corr_stable_hie,hybrid_corr_stable_hie,96,0.5,10,5,-3.622010,...,0.4711,0.0465,0.0993,5,5,2,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...,True,True,7


,bandit_name,set_name,family,strategy,z_window,alpha_out,n_market_features,n_seeds,seeds_used,median_of_seed_median_return_pct,...,median_profit_factor,median_total_trades,median_positive_return_symbols,median_active_symbols,median_long_action_symbols,all_symbols_active_share,all_symbols_have_long_share,feature_list,method_group,rank_within_bandit
0,discounted_lints,z24_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,hybrid_corr_stable_hie,24,0.5,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,-2.202887,...,0.847171,653.0,2.0,5.0,5.0,1.0,1.0,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,hybrid,1
1,discounted_lints,z96_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,hybrid_corr_stable_hie,96,0.5,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,-2.884910,...,0.797631,642.5,2.0,5.0,5.0,1.0,1.0,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...,hybrid,2
2,discounted_lints,z72_a0p5_corr_pruned_top10,corr_pruned,corr_pruned,72,0.5,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,-2.917490,...,0.825813,668.0,1.0,5.0,5.0,1.0,1.0,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...,corr,3
3,discounted_lints,z24_a0p5_corr_pruned_top10,corr_pruned,corr_pruned,24,0.5,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,-3.009619,...,0.812875,718.0,2.0,5.0,5.0,1.0,1.0,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,corr,4
4,discounted_lints,z96_a0p5_corr_pruned_top10,corr_pruned,corr_pruned,96,0.5,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,-4.498116,...,0.771833,617.0,1.0,5.0,5.0,1.0,1.0,range_sqrt_z|vol_ratio_24_log1p_z|vol_expansio...,corr,5
5,discounted_lints,z48_a0p5_corr_pruned_top10,corr_pruned,corr_pruned,48,0.5,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,-4.578935,...,0.727516,706.0,1.0,5.0,5.0,1.0,1.0,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...,corr,6
6,discounted_lints,z48_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,hybrid_corr_stable_hie,48,0.5,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,-4.721154,...,0.749091,596.0,1.0,5.0,5.0,1.0,1.0,range_sqrt_z|vol_ratio_72_log1p_z|MACD_hist_pc...,hybrid,7
7,discounted_lints,z72_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,hybrid_corr_stable_hie,72,0.5,10,30,142|143|144|145|146|147|148|149|150|151|152|15...,-5.196766,...,0.728347,666.0,1.0,5.0,5.0,1.0,1.0,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...,hybrid,8
8,discounted_linucb,z24_a0p5_hybrid_corr5_stablehie5_top10,hybrid_corr_stable_hie,hybrid_corr_stable_hie,24,0.5,10,1,142,1.041794,...,1.149639,491.0,3.0,5.0,5.0,1.0,1.0,ret_accel_6_24_z|volatility_72_sqrt_z|vol_rati...,hybrid,1
9,discounted_linucb,z72_a0p5_corr_pruned_top10,corr_pruned,corr_pruned,72,0.5,10,1,142,0.370538,...,1.022022,482.0,3.0,5.0,5.0,1.0,1.0,range_sqrt_z|vol_ratio_72_log1p_z|dist_to_high...,corr,2


Balanced selected feature sets by bandit:
{
  "discounted_lints": [
    {
      "set_name": "z72_a0p5_corr_pruned_top10",
      "method_group": "corr",
      "source": "top1_corr_within_discounted_lints_defaults_screening",
      "rank_within_bandit": 3,
      "median_of_seed_median_return_pct": -2.917489697332237,
      "mean_of_seed_median_return_pct": -2.78577348643189,
      "median_max_drawdown_abs": 11.07335420629888,
      "median_profit_factor": 0.8258131812274012,
      "median_total_trades": 668.0,
      "n_seeds": 30
    },
    {
      "set_name": "z24_a0p5_hybrid_corr5_stablehie5_top10",
      "method_group": "hybrid",
      "source": "top1_hybrid_within_discounted_lints_defaults_screening",
      "rank_within_bandit": 1,
      "median_of_seed_median_return_pct": -2.202887112072455,
      "mean_of_seed_median_return_pct": -1.9260867700939521,
      "median_max_drawdown_abs": 12.697959739033527,
      "median_profit_factor": 0.847171049831533,
      "median_total_trades": 65

,bandit_name,set_name,method_group,source,rank_within_bandit,median_of_seed_median_return_pct,mean_of_seed_median_return_pct,median_max_drawdown_abs,median_profit_factor,median_total_trades,...,hie_incremental_pool,n_hie_incremental_kept,n_corr_core_kept,n_corr_fallback_kept,n_hie_fallback_kept,hybrid_vs_corr_jaccard,hybrid_vs_corr_overlap_count,shared_features_with_corr,only_hybrid_vs_corr,only_corr_vs_hybrid
0,discounted_lints,z72_a0p5_corr_pruned_top10,corr,top1_corr_within_discounted_lints_defaults_scr...,3,-2.917490,-2.785773,11.073354,0.825813,668.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,discounted_lints,z24_a0p5_hybrid_corr5_stablehie5_top10,hybrid,top1_hybrid_within_discounted_lints_defaults_s...,1,-2.202887,-1.926087,12.697960,0.847171,653.0,...,"[ADX_30_scaled, dist_to_high_24_sqrt_z, price_...",5.0,5.0,0.0,0.0,0.333333,5.0,"[MOM_pct_72_z, range_sqrt_z, ret_accel_6_24_z,...","[ADX_30_scaled, RSI_72_bounded, dist_to_high_2...","[ADX_14_scaled, MOM_pct_30_z, NATR_14_z, ret_a..."
2,discounted_linucb,z72_a0p5_corr_pruned_top10,corr,top1_corr_within_discounted_linucb_defaults_sc...,2,0.370538,0.370538,10.489618,1.022022,482.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,discounted_linucb,z24_a0p5_hybrid_corr5_stablehie5_top10,hybrid,top1_hybrid_within_discounted_linucb_defaults_...,1,1.041794,1.041794,13.999347,1.149639,491.0,...,"[ADX_30_scaled, dist_to_high_24_sqrt_z, price_...",5.0,5.0,0.0,0.0,0.333333,5.0,"[MOM_pct_72_z, range_sqrt_z, ret_accel_6_24_z,...","[ADX_30_scaled, RSI_72_bounded, dist_to_high_2...","[ADX_14_scaled, MOM_pct_30_z, NATR_14_z, ret_a..."
4,sw_lints,z72_a0p5_corr_pruned_top10,corr,top1_corr_within_sw_lints_defaults_screening,2,-2.874477,-3.074464,11.151418,0.817973,624.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,sw_lints,z24_a0p5_hybrid_corr5_stablehie5_top10,hybrid,top1_hybrid_within_sw_lints_defaults_screening,1,-1.909723,-1.634037,10.255203,0.854427,544.5,...,"[ADX_30_scaled, dist_to_high_24_sqrt_z, price_...",5.0,5.0,0.0,0.0,0.333333,5.0,"[MOM_pct_72_z, range_sqrt_z, ret_accel_6_24_z,...","[ADX_30_scaled, RSI_72_bounded, dist_to_high_2...","[ADX_14_scaled, MOM_pct_30_z, NATR_14_z, ret_a..."
6,sw_linucb,z72_a0p5_corr_pruned_top10,corr,top1_corr_within_sw_linucb_defaults_screening,2,-1.154177,-1.154177,11.711811,0.866872,496.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,sw_linucb,z24_a0p5_hybrid_corr5_stablehie5_top10,hybrid,top1_hybrid_within_sw_linucb_defaults_screening,1,0.515848,0.515848,9.119914,1.008288,464.0,...,"[ADX_30_scaled, dist_to_high_24_sqrt_z, price_...",5.0,5.0,0.0,0.0,0.333333,5.0,"[MOM_pct_72_z, range_sqrt_z, ret_accel_6_24_z,...","[ADX_30_scaled, RSI_72_bounded, dist_to_high_2...","[ADX_14_scaled, MOM_pct_30_z, NATR_14_z, ret_a..."


## 10. Diagnostics and summary for the next Optuna stage

In [11]:
# Family/method comparison diagnostics: corr vs hybrid per bandit.
comparison_rows = []
for bandit_name, part in stage1_ranking.groupby("bandit_name", sort=False):
    corr_best = part[part["method_group"].eq("corr")].sort_values("rank_within_bandit").iloc[0]
    hybrid_best = part[part["method_group"].eq("hybrid")].sort_values("rank_within_bandit").iloc[0]
    comparison_rows.append({
        "bandit_name": bandit_name,
        "best_corr_set": corr_best["set_name"],
        "best_corr_rank": int(corr_best["rank_within_bandit"]),
        "best_corr_median_return_pct": float(corr_best["median_of_seed_median_return_pct"]),
        "best_corr_drawdown_abs": float(corr_best["median_max_drawdown_abs"]),
        "best_hybrid_set": hybrid_best["set_name"],
        "best_hybrid_rank": int(hybrid_best["rank_within_bandit"]),
        "best_hybrid_median_return_pct": float(hybrid_best["median_of_seed_median_return_pct"]),
        "best_hybrid_drawdown_abs": float(hybrid_best["median_max_drawdown_abs"]),
        "hybrid_minus_corr_return_pct": float(hybrid_best["median_of_seed_median_return_pct"] - corr_best["median_of_seed_median_return_pct"]),
        "winner_method_by_rank": "hybrid" if int(hybrid_best["rank_within_bandit"]) < int(corr_best["rank_within_bandit"]) else "corr",
    })

method_comparison = pd.DataFrame(comparison_rows)
method_comparison.to_csv(SCREENING_OUTPUT_DIR / "best_corr_vs_best_hybrid_by_bandit.csv", index=False)
display(method_comparison)

# Selected feature overlap between best corr and best hybrid per algorithm.
overlap_rows = []
for _, row in method_comparison.iterrows():
    bandit_name = row["bandit_name"]
    corr_set = row["best_corr_set"]
    hybrid_set = row["best_hybrid_set"]
    c = set(FEATURE_SETS[corr_set])
    h = set(FEATURE_SETS[hybrid_set])
    overlap_rows.append({
        "bandit_name": bandit_name,
        "corr_set": corr_set,
        "hybrid_set": hybrid_set,
        "corr_n": len(c),
        "hybrid_n": len(h),
        "intersection": len(c & h),
        "union": len(c | h),
        "jaccard": len(c & h) / max(len(c | h), 1),
        "shared_features": "|".join(sorted(c & h)),
        "only_corr": "|".join(sorted(c - h)),
        "only_hybrid": "|".join(sorted(h - c)),
    })

overlap_best_corr_hybrid = pd.DataFrame(overlap_rows)
overlap_best_corr_hybrid.to_csv(SCREENING_OUTPUT_DIR / "selected_best_corr_vs_hybrid_feature_overlap.csv", index=False)
display(overlap_best_corr_hybrid)

# Auto-summary for quick review.
summary_lines = []
summary_lines.append("Defaults screening summary: hybrid/corr alpha_out=0.5")
summary_lines.append(f"Feature sets evaluated: {len(screening_feature_sets)}")
summary_lines.append(f"Expected backtests: {expected_screening_backtests}")
summary_lines.append(f"Errors: {len(stage1_errors)}")
summary_lines.append("")
summary_lines.append("Balanced selected candidates for Optuna:")
for bandit_name, items in selected_feature_sets_by_bandit.items():
    summary_lines.append(f"  {bandit_name}:")
    for item in items:
        summary_lines.append(
            f"    {item['method_group']}: {item['set_name']} | "
            f"rank={item['rank_within_bandit']} | "
            f"median_return={item['median_of_seed_median_return_pct']:.4f}% | "
            f"drawdown={item['median_max_drawdown_abs']:.4f}"
        )
summary_lines.append("")
summary_lines.append("Best corr vs best hybrid by bandit:")
for _, row in method_comparison.iterrows():
    summary_lines.append(
        f"  {row['bandit_name']}: winner_by_rank={row['winner_method_by_rank']}, "
        f"hybrid_minus_corr_return={row['hybrid_minus_corr_return_pct']:.4f}%"
    )

summary_text = "\n".join(summary_lines)
(SCREENING_OUTPUT_DIR / "screening_summary_for_next_stage.txt").write_text(summary_text, encoding="utf-8")
print(summary_text)

,bandit_name,best_corr_set,best_corr_rank,best_corr_median_return_pct,best_corr_drawdown_abs,best_hybrid_set,best_hybrid_rank,best_hybrid_median_return_pct,best_hybrid_drawdown_abs,hybrid_minus_corr_return_pct,winner_method_by_rank
0,discounted_lints,z72_a0p5_corr_pruned_top10,3,-2.917490,11.073354,z24_a0p5_hybrid_corr5_stablehie5_top10,1,-2.202887,12.697960,0.714603,hybrid
1,discounted_linucb,z72_a0p5_corr_pruned_top10,2,0.370538,10.489618,z24_a0p5_hybrid_corr5_stablehie5_top10,1,1.041794,13.999347,0.671256,hybrid
2,sw_lints,z72_a0p5_corr_pruned_top10,2,-2.874477,11.151418,z24_a0p5_hybrid_corr5_stablehie5_top10,1,-1.909723,10.255203,0.964754,hybrid
3,sw_linucb,z72_a0p5_corr_pruned_top10,2,-1.154177,11.711811,z24_a0p5_hybrid_corr5_stablehie5_top10,1,0.515848,9.119914,1.670025,hybrid


,bandit_name,corr_set,hybrid_set,corr_n,hybrid_n,intersection,union,jaccard,shared_features,only_corr,only_hybrid
0,discounted_lints,z72_a0p5_corr_pruned_top10,z24_a0p5_hybrid_corr5_stablehie5_top10,10,10,1,19,0.052632,range_sqrt_z,ADX_14_scaled|MACD_hist_pct_z|MOM_pct_14_z|NAT...,ADX_30_scaled|MOM_pct_72_z|RSI_72_bounded|dist...
1,discounted_linucb,z72_a0p5_corr_pruned_top10,z24_a0p5_hybrid_corr5_stablehie5_top10,10,10,1,19,0.052632,range_sqrt_z,ADX_14_scaled|MACD_hist_pct_z|MOM_pct_14_z|NAT...,ADX_30_scaled|MOM_pct_72_z|RSI_72_bounded|dist...
2,sw_lints,z72_a0p5_corr_pruned_top10,z24_a0p5_hybrid_corr5_stablehie5_top10,10,10,1,19,0.052632,range_sqrt_z,ADX_14_scaled|MACD_hist_pct_z|MOM_pct_14_z|NAT...,ADX_30_scaled|MOM_pct_72_z|RSI_72_bounded|dist...
3,sw_linucb,z72_a0p5_corr_pruned_top10,z24_a0p5_hybrid_corr5_stablehie5_top10,10,10,1,19,0.052632,range_sqrt_z,ADX_14_scaled|MACD_hist_pct_z|MOM_pct_14_z|NAT...,ADX_30_scaled|MOM_pct_72_z|RSI_72_bounded|dist...


Defaults screening summary: hybrid/corr alpha_out=0.5
Feature sets evaluated: 8
Expected backtests: 496
Errors: 0

Balanced selected candidates for Optuna:
  discounted_lints:
    corr: z72_a0p5_corr_pruned_top10 | rank=3 | median_return=-2.9175% | drawdown=11.0734
    hybrid: z24_a0p5_hybrid_corr5_stablehie5_top10 | rank=1 | median_return=-2.2029% | drawdown=12.6980
  discounted_linucb:
    corr: z72_a0p5_corr_pruned_top10 | rank=2 | median_return=0.3705% | drawdown=10.4896
    hybrid: z24_a0p5_hybrid_corr5_stablehie5_top10 | rank=1 | median_return=1.0418% | drawdown=13.9993
  sw_lints:
    corr: z72_a0p5_corr_pruned_top10 | rank=2 | median_return=-2.8745% | drawdown=11.1514
    hybrid: z24_a0p5_hybrid_corr5_stablehie5_top10 | rank=1 | median_return=-1.9097% | drawdown=10.2552
  sw_linucb:
    corr: z72_a0p5_corr_pruned_top10 | rank=2 | median_return=-1.1542% | drawdown=11.7118
    hybrid: z24_a0p5_hybrid_corr5_stablehie5_top10 | rank=1 | median_return=0.5158% | drawdown=9.1199

Best 